# 🎵 Sound Metadata Extractor — Colab Launcher (Optimized)

> **Code source:** [`ahmedHosney600/Sounds-Meta-Info-Extractor`](https://github.com/ahmedHosney600/Sounds-Meta-Info-Extractor)  
> **Data source:** Your Google Drive sounds folder  
> **Outputs:** Saved back to Google Drive

---
### Run order
1. **Cell 1** — Mount Google Drive (for sounds + outputs)
2. **Cell 2** — Clone project from GitHub & install dependencies
3. **Cell 3a** — ⚙️ **Set your configuration** (edit only this cell)
4. **Cell 3b** — 🔗 **Generate Drive Links Map** (fixes API crash)
5. **Cell 4** — ▶️ **Run the extractor (Multiprocessing)**
6. **Cell 5** — (Optional) Preview results table

In [ ]:
# ============================================================
# CELL 1 — Mount Google Drive
# ============================================================
from google.colab import drive
drive.mount('/content/drive')
print('✅ Google Drive mounted at /content/drive')

In [ ]:
# ============================================================
# CELL 2 — Clone / Update from GitHub & Install Dependencies
# ============================================================
import os, sys

REPO_URL  = 'https://github.com/ahmedHosney600/Sounds-Meta-Info-Extractor.git'
CLONE_DIR = '/content/Sounds-Meta-Info-Extractor'

if os.path.exists(CLONE_DIR):
    print('🔄 Repo already cloned — resetting to latest GitHub version...')
    !git -C "{CLONE_DIR}" fetch origin
    !git -C "{CLONE_DIR}" reset --hard origin/main
else:
    print('📥 Cloning repository from GitHub...')
    !git clone "{REPO_URL}" "{CLONE_DIR}"

# Confirm which commit is loaded
!git -C "{CLONE_DIR}" log --oneline -3

# System dependency: ffmpeg for AIFF, M4A, OGG, OPUS, etc.
print('\nInstalling ffmpeg...')
!apt-get install -y ffmpeg > /dev/null 2>&1
print('✅ ffmpeg installed')

# Python dependencies
print('Installing Python libraries...')
!pip install -q -r "{CLONE_DIR}/requirements.txt"
print('✅ All libraries installed')

# Add project to Python path
if CLONE_DIR not in sys.path:
    sys.path.insert(0, CLONE_DIR)
print(f'✅ Project ready: {CLONE_DIR}')

In [ ]:
# ============================================================
# CELL 3a — ⚙️ CONFIGURATION  (only edit this cell)
# ============================================================
import os

# ── Sounds folder on Google Drive ──────────────────────────────────
# Full path to your sounds folder as mounted in Colab.
# Example: '/content/drive/MyDrive/Sound Libraries/BLOW'
os.environ['SOUNDS_FOLDER'] = '/content/drive/MyDrive/BOOOM FINAL'

# ── Google Drive Folder ID (for preview/download links) ─────────
# Open your sounds folder in Drive in the browser.
# The URL will be: https://drive.google.com/drive/folders/<ID>
# Paste the ID below. Leave empty to skip link generation.
os.environ['DRIVE_FOLDER_ID'] = ''  # e.g. '1AbCdEfGhIjKlMnOpQr'

# ── Output folder (saved to your Google Drive) ────────────────
os.environ['OUTPUT_FOLDER'] = '/content/sounds_metadata_output'

# ── Scan & Performance Settings ─────────────────────────────────
os.environ['RECURSIVE']        = 'true'   # search subfolders?
os.environ['MAX_FILE_SIZE_MB'] = '1000'   # skip files > this MB (0 = no limit)
os.environ['NUM_WORKERS']      = '4'      # number of parallel extraction threads

print('✅ Configuration:')
print(f'   SOUNDS_FOLDER   = {os.environ["SOUNDS_FOLDER"]}')
print(f'   DRIVE_FOLDER_ID = {os.environ["DRIVE_FOLDER_ID"] or "(not set — Drive links disabled)"}')
print(f'   OUTPUT_FOLDER   = {os.environ["OUTPUT_FOLDER"]}')
print(f'   MAX_FILE_SIZE   = {os.environ["MAX_FILE_SIZE_MB"]} MB')
print(f'   NUM_WORKERS     = {os.environ["NUM_WORKERS"]}')

In [ ]:
# ============================================================
# CELL 3b — 🔗 Generate Drive Links Map
# ============================================================
# Run this cell to authenticate and build a map of your files.
# This avoids the subprocess authentication crash in Cell 4.
import os
import json

folder_id = os.environ.get('DRIVE_FOLDER_ID')
os.environ['DRIVE_ID_MAP_FILE'] = '/content/drive_id_map.json'

if folder_id:
    print('🔑 Authenticating with Google (requires popup approval)...')
    from google.colab import auth as colab_auth
    colab_auth.authenticate_user()

    print('
🔎 Querying Google Drive API (this might take a minute for large folders)...')
    from googleapiclient.discovery import build
    import google.auth
    from extractors.drive_links import build_drive_id_map

    creds, _ = google.auth.default()
    drive_service = build('drive', 'v3', credentials=creds)
    
    is_recursive = os.environ.get('RECURSIVE', 'true').lower() == 'true'
    id_map = build_drive_id_map(folder_id, drive_service, recursive=is_recursive)
    
    with open(os.environ['DRIVE_ID_MAP_FILE'], 'w') as f:
        json.dump(id_map, f)
    print(f'✅ Saved Drive map with {len(id_map)} files to {os.environ["DRIVE_ID_MAP_FILE"]}')
else:
    print('⚠️ DRIVE_FOLDER_ID not set in Cell 3a. Skipping map generation.')

In [ ]:
# ============================================================
# CELL 4 — ▶️ Run the Extractor (Multiprocessing)
# ============================================================
CLONE_DIR = '/content/Sounds-Meta-Info-Extractor'

# Run main.py from the cloned repo.
# Environment variables set in Cell 3a/3b are automatically inherited.
!cd "{CLONE_DIR}" && python main.py

In [ ]:
# ============================================================
# CELL 5 — (Optional) Preview Results Table
# ============================================================
import json, os
import pandas as pd
from IPython.display import display

output_folder = os.environ.get('OUTPUT_FOLDER', '/content/sounds_metadata_output')
json_path = f'{output_folder}/sounds.json'

with open(json_path, 'r', encoding='utf-8') as f:
    data = json.load(f)

print(f'Loaded {len(data)} records from {json_path}')

PREVIEW_COLS = [
    'filename', 'extension', 'sf_duration_seconds', 'sf_sample_rate',
    'sf_bit_depth', 'sf_channel_label', 'lb_tempo_bpm',
    'ai_top_class', 'ai_top_score', 'heuristic_sound_type',
    'fn_parsed_category', 'fn_parsed_description',
    'drive_preview_url',
]

def flatten(r):
    return {k: (str(v) if isinstance(v, (list, dict)) else v) for k, v in r.items()}

df = pd.DataFrame([flatten(r) for r in data])
available = [c for c in PREVIEW_COLS if c in df.columns]
display(df[available].head(20))